# Lesson 1.3.3 - Basic REST APIs with HTTP & JSON

## Learning Objectives
- Build a small JSON API handler with explicit schema validation.
- Implement a resilient client retry pattern for transient failures.
- Use practical API readiness checklists for ML/AI services.


## API Contract First
Before coding endpoints, define:
- request schema
- response schema
- status code behavior
- idempotency and retry policy


In [1]:
from dataclasses import dataclass
from typing import Any


@dataclass
class PredictionRequest:
    age: float
    income: float


def validate_payload(payload: dict[str, Any]) -> PredictionRequest:
    required = {'age', 'income'}
    missing = required - payload.keys()
    if missing:
        raise ValueError(f'missing fields: {sorted(missing)}')
    age = float(payload['age'])
    income = float(payload['income'])
    if age < 0 or income < 0:
        raise ValueError('age/income must be non-negative')
    return PredictionRequest(age=age, income=income)


def predict(req: PredictionRequest) -> dict[str, Any]:
    score = req.age * 0.02 + req.income * 0.00001
    label = int(score > 0.8)
    return {'score': round(score, 4), 'label': label, 'model_version': 'v1.0'}


def handle_request(payload: dict[str, Any]) -> tuple[int, dict[str, Any]]:
    try:
        req = validate_payload(payload)
        return 200, predict(req)
    except ValueError as exc:
        return 400, {'error': str(exc)}


print(handle_request({'age': 34, 'income': 90000}))
print(handle_request({'age': -1, 'income': 50000}))


(200, {'score': 1.58, 'label': 1, 'model_version': 'v1.0'})
(400, {'error': 'age/income must be non-negative'})


## Reusable Resilient Client Template
Use bounded retries only for retryable failures (e.g., 429/503).


In [2]:
import time


class TransientAPIError(RuntimeError):
    pass


def flaky_backend(counter: dict) -> dict:
    counter['calls'] += 1
    if counter['calls'] < 3:
        raise TransientAPIError('503 temporary outage')
    return {'ok': True, 'attempt': counter['calls']}


def call_with_retry(max_retries: int = 3, base_delay: float = 0.1) -> dict:
    state = {'calls': 0}
    for attempt in range(max_retries + 1):
        try:
            return flaky_backend(state)
        except TransientAPIError as exc:
            if attempt == max_retries:
                raise
            time.sleep(base_delay * (2 ** attempt))
    raise RuntimeError('unreachable')


print(call_with_retry())


{'ok': True, 'attempt': 3}


## API Readiness Checklist
- [ ] Request and response JSON schemas are explicit.
- [ ] All non-2xx paths have machine-readable errors.
- [ ] Timeouts are configured in clients.
- [ ] Retry policy excludes non-retryable failures (e.g., validation errors).
- [ ] Logging excludes PII and secrets.
- [ ] Model version is returned for traceability.


## Business Case Studies & Exceptions
### Case 1: Prediction API Outage Cascade
No timeout/retry guard caused worker saturation when a dependency slowed. Fix: strict timeout + capped retries + fallback response mode.

### Case 2: Silent Contract Drift
Upstream service renamed `score` field. Downstream analytics silently broke. Fix: response schema validation and integration tests for API contracts.

### Exception
For local-only batch scripts, full REST stack may be unnecessary. A CLI job with file-based contracts can be simpler and more reliable.


## Interview Questions & Answers
1. **Q:** Why define JSON schema up front?  
   **A:** It prevents ambiguous contracts and reduces integration bugs.
2. **Q:** What is idempotency in API design?  
   **A:** Repeating a request yields the same effect, enabling safer retries.
3. **Q:** Which errors are generally retryable?  
   **A:** Transient errors like rate limit or temporary server unavailability.
4. **Q:** Why include `model_version` in responses?  
   **A:** For debugging, auditing, and rollback traceability.
5. **Q:** What does status code 400 mean?  
   **A:** Client-side request error (invalid payload or parameters).
6. **Q:** What is a key API security rule?  
   **A:** Never log secrets/PII and avoid hardcoded credentials.
7. **Q:** Why not retry 4xx validation errors?  
   **A:** They are usually deterministic client mistakes, not transient failures.
8. **Q:** Why set client timeouts explicitly?  
   **A:** Prevent hanging requests and cascading failures.
9. **Q:** What should an API error payload include?  
   **A:** Error code/type, message, and optional request trace ID.
10. **Q:** REST vs RPC for ML inference?  
   **A:** REST is simpler for many teams; RPC may be preferred for strict low-latency contracts at scale.
